In [ ]:
!pip install -q langgraph langchain-core langchain-google-genai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.6/81.6 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 22.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 262.4/262.4 kB 18.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.58.0 which is incompatible.


In [1]:
import langgraph, langchain_core
from importlib.metadata import version
print("langgraph:", version("langgraph"))
print("langchain-core:", langchain_core.__version__)
print("langchain-google-genai:", version("langchain-google-genai"))

langgraph: 1.2.11
langchain-core: 1.6.3
langchain-google-genai: 4.4.0


In [2]:
import langgraph, langchain_core
from dotenv import load_dotenv
import os

load_dotenv()
print("langgraph OK, dotenv OK")
print("key loaded:", os.getenv("RESEARCHER_API_KEY") is not None)

langgraph OK, dotenv OK
key loaded: True


In [3]:
# Loading API Key For Colab
# from google.colab import userdata
# import os
# key_names = ["Researcher", "Writer", "Critic"]
# for name in key_names:
#   val = userdata.get(name)
#   os.environ[name] = val
#   print(f"{name}: {'OK, len=' + str(len(val)) if val else 'MISSING'}")

In [4]:
from typing import TypedDict, List
class ResearchState(TypedDict):
  topic: str            # User input topic
  research_notes: str   # Now: Clean plain text after extration (from extract_research_notes)
                        # Pre:Researcher generate info clearn up
  research_rounds: int  # Adding Researcher actual search round (from coutn_research_rounds)
  draft: str            # Writer current draft
  critique: str         # Critic latest judge opinion
  revision_count: int   # Current correctness times, for controling loop ending
  approved: bool        # Critic passing or not

print("State Def result complete")

State Def result complete


In [16]:
# Loading Gemini Model
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_tavily import TavilySearch

researcher_base_llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite",
    google_api_key=os.environ["RESEARCHER_API_KEY"],
    timeout=30,
    max_retries=1,
)

# Writer & Critic ideally use 3.5-flash
#  Due to the daily 20 limitation for 3.5-flash
#  , use 3.5-flash-lite instead

# writer_llm = ChatGoogleGenerativeAI(
#     model="gemini-3.5-flash",
#     google_api_key=os.environ["Writer"],
# )

# critic_llm = ChatGoogleGenerativeAI(
#     model="gemini-3.5-flash",
#     google_api_key=os.environ["Critic"],
# )

writer_llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite",
    google_api_key=os.environ["WRITER_API_KEY"],
    timeout=30,
    max_retries=1,
)

critic_llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite",
    google_api_key=os.environ["CRITIC_API_KEY"],
    timeout=30,
    max_retries=1,
)

web_search = TavilySearch(
    max_results=3,
    tavily_api_key=os.getenv("TAVILY_API_KEY"),
)

researcher_llm_with_tools = researcher_base_llm.bind_tools([web_search])


In [6]:
# Assistant Function

def extract_text(response) -> str:
  """
    Dealing with Gemini return format (new version could be list of dict, or number)
  """
  content = response.content
  if isinstance(content, str):
    return content
  if isinstance(content, list):
    return "".join(
        block.get("text", "")
        for block in content
        if isinstance(block,dict)
    )
  return str(content)

In [8]:
from langchain_core.messages import HumanMessage, ToolMessage, SystemMessage

# System_prompt setup
DEFAULT_SYSTEM_PROMPT = SystemMessage(content=(
    "You are a research assistant. You have a web_search tool that can "
    "check real-time data. If the information you have is enough to "
    "answer the user's question, answer directly with text -- do not "
    "call the tool again. Only call the tool again if the current "
    "information is clearly insufficient, conflicting, or missing key details."
))

MAX_ITERATIONS = 3 # decided from Trail 5: real-data trails stopped at round 2;

def researcher_loop(question: str, system_prompt: SystemMessage = DEFAULT_SYSTEM_PROMPT) -> list:
    """
    Runs the Researcher's tool-calling loop until the LLM decides it has enough information,
    or MAX_ITERATIONS is hit (safety net).
    Returns the full message history for downstream use (Writer)
    """

    messages = [
        system_prompt,
        HumanMessage(content=question),
    ]

    for round_num in range(1, MAX_ITERATIONS + 1):
        response = researcher_llm_with_tools.invoke(messages)
        messages.append(response)

        if not response.tool_calls:
            # LLM judged it has enough info: stop
            return messages

        for tc in response.tool_calls:
            result = web_search.invoke(tc["args"])
            messages.append(ToolMessage(content=str(result), tool_call_id=tc["id"]))

    # Hit the limit
    return messages

# First test "Open Question" (The question that will trigger tool recall)
print("=== Open-ended question ===")
researcher_loop("What is the weather today in San Jose, CA?")

=== Open-ended question ===


[SystemMessage(content="You are a research assistant. You have a web_search tool that can check real-time data. If the information you have is enough to answer the user's question, answer directly with text -- do not call the tool again. Only call the tool again if the current information is clearly insufficient, conflicting, or missing key details.", additional_kwargs={}, response_metadata={}),
 HumanMessage(content='What is the weather today in San Jose, CA?', additional_kwargs={}, response_metadata={}),
 AIMessage(content=[], additional_kwargs={'function_call': {'name': 'tavily_search', 'arguments': '{"query": "weather today San Jose CA"}'}, '__gemini_function_call_thought_signatures__': {'call_22597': 'El4KXAFpFH0TIw4Y238FKsudHdgdJJqJWTNdDisD7wwATIvhucOX+eUtuw6tO1aSTihbh2uqRwBJuA/Hy8FlE2sactDXP0u+ouDT8+77MhUL6M9z6eiGGsM9tpu10eP2'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run-

In [9]:
import json
import ast

def extract_research_notes(messages: list) -> str:
    """
    Extract all ToolMessage's search result from researcher_loop returns message.
    Only keep title and content with plain text for Writer.
    """
    notes_parts = []

    for msg in messages:
        if not isinstance(msg, ToolMessage):
            continue

        raw = msg.content
        # ToolMessage.content is Tavily return dict.
        try:
            data = ast.literal_eval(raw) if isinstance(raw, str) else raw
        except Exception:
            # If extract fail, it will be kept as plain text
            notes_parts.append(str(raw))
            continue

        results = data.get("results", [])
        for r in results:
            title = r.get("title", "")
            content = r.get("content", "")
            notes_parts.append(f"[{title}] {content}")

    return "\n\n".join(notes_parts)


# Verify: Use previous result
test_messages = researcher_loop("What is the weather today in San Jose, CA?")
notes = extract_research_notes(test_messages)
print(notes[:500])
print("---")
print(f"Total length: {len(notes)} chars")

[Weather in San Jose, CA] {'location': {'name': 'San Jose', 'region': 'California', 'country': 'United States of America', 'lat': 37.3394, 'lon': -121.8939, 'tz_id': 'America/Los_Angeles', 'localtime_epoch': 1790091805, 'localtime': '2026-09-22 08:43'}, 'current': {'last_updated_epoch': 1790091000, 'last_updated': '2026-09-22 08:30', 'temp_c': 12.6, 'temp_f': 54.8, 'is_day': 1, 'condition': {'text': 'Sunny', 'icon': '//cdn.weatherapi.com/weather/64x64/day/113.png', 'code': 1000}, 'wind_mph': 3.1
---
Total length: 3152 chars


In [10]:
def count_research_rounds(messages: list) -> int:
    """
    Calculate the actual web_search tool calling count
    The value is same as ToolMessage's number
    The nubmer will be store in State["research_rounds"] for further usage.    
    """
    return sum(1 for msg in messages if isinstance(msg, ToolMessage))


# Verify
print(count_research_rounds(test_messages))

1


In [11]:
# Research Node Define
def researcher_node(state: ResearchState) -> dict:
  """
  Replace previous single node LLM calling Researcher
  Change to researcher_loop for Researcher decide if it need to recheck
  Then use extract_research_notes / count_research_rounds convert State needed form
  """
  messages = researcher_loop(state["topic"])
  return {
    "research_notes": extract_research_notes(messages),
    "research_rounds": count_research_rounds(messages),
    }

# Quick test
test_state = {"topic": "Current Taiwan electrical motorbike situation",
              "research_notes": "",
              "research_rounds": 0,
              "draft": "",
              "critique": "",
              "revision_count": 0,
              "approved": False}
result = researcher_node(test_state)
print(type(result["research_notes"]))
print(result["research_notes"][:200])


<class 'str'>
[Gogoro’s Reset: From Electric Scooter Brand to Energy Infrastructure Company - CleanTechnica] ### The test ahead

Investor skepticism remains high. Gogoro’s share price reflects a survival narrative 


In [12]:
# Writer Node Define
def writer_node(state: ResearchState) -> dict:
  if state.get("critique"):
    # Modification Loop: regarding to Critic to modify previous draft
    prompt = (
        f"This is the draft that you wrote regarding to the topic '{state['topic']}'"
        f"Critiquer gave the opinion as follow: \n{state['critique']}\n\n"
        f"Please modify the draft, and output the corrected result."
        "Not only what you changed, but the complete draft"
    )
  else:
    # First Loop: draft according to the topic
    prompt = (
        f"You are a professional writer. Regarding to the topic: '{state['topic']}'"
        f"Write a clear structured, 300-500 words first draft: \n\n{state['research_notes']}"
    )
  response = writer_llm.invoke(prompt)
  return {
      "draft": extract_text(response),
      "revision_count": state["revision_count"] + (1 if state.get("critique") else 0),
  }

# Quick test follow the previous research_notes
test_state["research_notes"] = result["research_notes"]
draft_result = writer_node(test_state)
print(draft_result["draft"][:300])
print("revision_count:", draft_result["revision_count"])

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


**Title:** Shifting Gears: Taiwan’s Electric Motorbike Market Pivots from Vehicle Sales to Energy Infrastructure

Taiwan’s electric two-wheeler sector is undergoing a profound structural transformation. Once celebrated as a global pioneer in green urban mobility, the market has faced a protracted sl
revision_count: 0


In [13]:
# Critic Node Define
def critic_node(state: ResearchState) -> dict:
    prompt = (
        f"You are a strict checker, check the report draft regarding to the topic:'{state['topic']}':\n\n"
        f"{state['draft']}\n\n"
        f"Check if content is accurate, clear, or if there's any obvious missing point or logic issue.\n"
        f"Please use the following format to reply (first line must be APPROVED or REVISE, not other word in the first line)\n"
        f"APPROVED or REVISE\n"
        f"The following should start the opinion (if it is APPROVED, brifly explain why)"
    )
    response = critic_llm.invoke(prompt)
    text = extract_text(response).strip()

    lines = text.split("\n", 1)
    verdict = lines[0].strip().upper()
    feedback = lines[1].strip() if len(lines) > 1 else ""

    return {
        "approved": verdict.startswith("APPROVED"),
        "critique": feedback,
        "revision_count": state["revision_count"]
    }

# Quick test: use draft from previous quick test
test_state["draft"] = draft_result["draft"]
critique_result = critic_node(test_state)
print("approved:", critique_result["approved"])
print("critique:", critique_result["critique"][:300])

approved: False
critique: **Critical Analysis of the Draft:**

While the draft is well-written, engaging, and features a strong central thesis, it contains **a major chronological logic flaw** regarding the year 2025 that must be corrected.

**Specific Issues Found:**
1. **The 2025 Timeline Paradox:** 
   * In paragraph 2, t


In [14]:
from langgraph.graph import StateGraph, START, END

MAX_REVISIONS = 3  # Temporary early stop

# Condition edge for critic
def route_after_critic(state: ResearchState) -> str:
    if state["approved"] or state["revision_count"] >= MAX_REVISIONS:
        return "end"
    return "revise"

graph_builder = StateGraph(ResearchState)
# add three nodes for researcher, writer, and critic
graph_builder.add_node("researcher", researcher_node)
graph_builder.add_node("writer", writer_node)
graph_builder.add_node("critic", critic_node)

# add edges connect three state nodes
graph_builder.add_edge(START, "researcher")
graph_builder.add_edge("researcher", "writer")
graph_builder.add_edge("writer", "critic")

# If approved or reach MAX_REVISIONS -> END state
# Eles -> revise
graph_builder.add_conditional_edges(
    "critic",
    route_after_critic,
    {"end": END, "revise": "writer"},
)

graph = graph_builder.compile()
print("Graph Complete Edit")

Graph Complete Edit


In [17]:
initial_state = {
    "topic": "Latest developments in humanoid robotics 2026",
    "research_notes": "",
    "research_rounds": 0,
    "draft": "",
    "critique": "",
    "revision_count": 0,
    "approved": False,
}

final_state = graph.invoke(initial_state)

print("=== Result ===")
print("Approved:", final_state["approved"])
print("Revision count:", final_state["revision_count"])
print("Research rounds:", final_state["research_rounds"])   # 新欄位，確認有正確傳遞through整個 graph
print("\n=== Research Notes (first 300 chars) ===")
print(final_state["research_notes"][:300])
print("\n=== Final Draft (first 300 chars) ===")
print(final_state["draft"][:300])

GoogleAPIError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}

In [ ]:
# Testing model issue
# try:
#     test = researcher_base_llm.invoke("Say hello in one word.")
#     print("OK:", extract_text(test))
# except Exception as e:
#     print("FAILED:", e)

FAILED: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}


In [43]:
initial_state = {
    "topic": "AI Agent in Enterprise, how employee prevent to be laid off",
    "research_notes": "",
    "draft": "",
    "critique": "",
    "revision_count": 0,
    "approved": False,
}

for event in graph.stream(initial_state):
    for node_name, node_output in event.items():
        print(f"--- State: {node_name} ---")
        if "approved" in node_output:
            print(f"  approved={node_output['approved']}, revision_count={node_output['revision_count']}")
        elif "draft" in node_output:
            print(f"  draft (First 80 char): {node_output['draft'][:80]}...")
        elif "research_notes" in node_output:
            print(f"  research_notes (First 80 char): {node_output['research_notes'][:80]}...")
        print()

--- State: researcher ---
  research_notes (First 80 char): ...

--- State: writer ---
  draft (First 80 char): **Title:** Beyond the Bot: How Employees Can Future-Proof Their Careers in the A...

--- State: critic ---
  approved=True, revision_count=0



In [44]:
def run_research_pipeline(topic: str) -> dict:
    initial_state = {
        "topic": topic,
        "research_notes": "",
        "draft": "",
        "critique": "",
        "revision_count": 0,
        "approved": False,
    }
    final_state = graph.invoke(initial_state)
    return {
        "topic": topic,
        "final_draft": final_state["draft"],
        "approved": final_state["approved"],
        "revision_count": final_state["revision_count"],
        "final_critique": final_state["critique"],
    }

# Testing
output = run_research_pipeline("How junior SWE can get hired after laid off in 2026")
print("approved:", output["approved"])
print("revision_count:", output["revision_count"])
print(output["final_draft"][:200])

approved: False
revision_count: 3
Here is the complete, revised draft incorporating the critic’s feedback: the "mid-level" contradiction has been resolved to accurately reflect market realities for an experienced junior, and a crucial


In [15]:
# Use gradio to build a simple UI
import gradio as gr

def gradio_handler(topic):
    if not topic.strip():
        return "Please enter the research topic", "", ""
    output = run_research_pipeline(topic)
    status = "✅ Approved" if output["approved"] else "⚠️ Meet the max revise rounds, Not Approved"
    meta = f"{status}| revision count:{output['revision_count']}"
    return meta, output["final_draft"], output["final_critique"]

demo = gr.Interface(
    fn=gradio_handler,
    inputs=gr.Textbox(label="Research Topic", placeholder="ex: Battery factory in 2026"),
    outputs=[
        gr.Textbox(label="Status"),
        gr.Markdown(label="Final Report"),
        gr.Textbox(label="Critic Final Opinion"),
    ],
    title="Multi-Agent Research Assistants",
    description="Researcher → Writer → Critic Cooperate Research report (Gemini 3.5 Flash-Lite)",
)

demo.launch(debug=True)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


Keyboard interruption in main thread... closing server.
